# Preparation

In [ ]:
import os
import sys

# Go up one directory level to the project root folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
print(project_root)

if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can successfully import from your module folder


In [ ]:
from module.DepthConstSet import DepthConstSet
from module.NyuDatasetV2 import *
from module.MLModel import SegFormerDepth
from module.Training import train_model, plot_metrics, test_model

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt # For visualization


In [ ]:
# Ensure reproducibility for consistent results
torch.manual_seed(42)
np.random.seed(42)

# Define the path to your NYU Depth V2 labeled dataset (.mat file).
# You MUST replace this with the actual path on your system.
# The dataset can typically be downloaded from the official NYU Depth V2 website
# or other academic sources.
NYU_DATASET_PATH = 'nyu_depth_v2_labeled.mat' 
nyuv2_path = project_root + "/module/nyuv2_python_toolkit_master/NYUv2"

# Define the number of classes for segmentation.
# For NYU Depth V2, a common setup uses 40 semantic classes + 1 for unlabeled/background (label 0).
# So, total classes = 41.
NUM_CLASSES = 13
BATCH_SIZE = 8
LR = 5e-4
WEIGHT_DECAY = 1e-3

# Define the target image size for resizing. NYU images are 480x640.
# Downsampling can speed up training.
# IMAGE_SIZE = (240, 320) 
# IMAGE_SIZE = (480, 640) 
IMAGE_SIZE = (480, 480) 

EPOCH = 100

In [ ]:
# Set the device for training (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset

## Set Dataloader

In [ ]:
train_loader, val_loader, test_loader = get_train_val_dataloaders(
    dataset_path=nyuv2_path,
    batch_size=BATCH_SIZE,         # Small batch size for demonstration; increase for real training
    image_size=IMAGE_SIZE,
    transform_depth = False
)

In [ ]:
first_batch = next(iter(train_loader))
# image, (depth, label) = first_batch
(img, depthDict, GT) = first_batch

In [ ]:
depthDict["raw"].shape

In [ ]:
depthDict.keys()

In [ ]:
depthDict[DepthConstSet().raw].shape

In [ ]:
GT.shape

# Visualize dataset

In [ ]:
# --- Visualize a batch from the train_loader ---
print("\nVisualizing a batch from the training DataLoader...")
# Get one batch
# first_batch = next(iter(train_loader))


In [ ]:
# Categories
# train_loader.dataset.dataset.__getLabelList__()

In [ ]:
(rgb_images, depth_maps, labels) = first_batch

In [ ]:
# To denormalize RGB images correctly, we need access to the NyuDataset instance's
# mean and std. We can get this from the `dataset` attribute of the DataLoader's
# `dataset` attribute (which is a Subset object from random_split).
# The `dataset` attribute of the `Subset` object holds the original `NyuDataset`.

visualize_batch(first_batch, NUM_CLASSES, transform_depth=False)
print("Visualization complete. A plot window should have appeared.")
# --- End Visualization ---


# Model

In [ ]:
modelName = "SegFormerDepth"
in_channel=3
modelRaw = SegFormerDepth(in_channels=in_channel, num_classes=NUM_CLASSES, image_size=IMAGE_SIZE)
modelRaw.eval()

In [ ]:
x = torch.randn((BATCH_SIZE, in_channel, IMAGE_SIZE[0], IMAGE_SIZE[1]))
x.shape

In [ ]:
x = torch.randn((BATCH_SIZE, in_channel, IMAGE_SIZE[0], IMAGE_SIZE[1]))
# model = UNetModel(in_channels=4, num_classes=NUM_CLASSES)
with torch.no_grad():
    outputs = modelRaw(x)
# logits = outputs.logits  # [B, num_classes, h, w]


In [ ]:
print(f"x shape: {x.shape}")
print(f"pred shape: {outputs.shape}")
predShape=torch.Size([BATCH_SIZE, NUM_CLASSES, IMAGE_SIZE[0], IMAGE_SIZE[1]])
print(f"expected shape: {predShape}")
assert outputs.shape == predShape

# Training Depth Raw

## Training

In [ ]:
model_save_path = modelName + "_depth_segmentation_RGB.pth" # Define path for saved model# NUM_CLASSES=14
train_losses, train_accuracies, train_mious, val_losses, val_accuracies, val_mious = train_model(
    model=modelRaw,
    train_dataloader=train_loader,
    val_dataloader=val_loader, 
    num_classes=NUM_CLASSES,
    epochs=EPOCH, # Small number of epochs for demonstration; increase for actual training
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    device=device,
    model_save_path=model_save_path,
    RGB_ToTrain=True,
    PD_ToTrain = DepthConstSet().raw
    # use_logits = True
)

In [ ]:
# Plot the collected metrics after training
print("\n--- Plotting Training and Validation Metrics ---")
plot_metrics(train_losses, train_accuracies, train_mious, 
                val_losses, val_accuracies, val_mious, epochs=EPOCH)
print("Metric plots generated.")

## Testing

In [ ]:
# --- Test the trained model ---
print("\n--- Running Test Model ---")
# You can use val_loader as a proxy for a test set for this example
test_model(
    model=modelRaw,
    test_dataloader=test_loader, # Using test loader for demonstration
    num_classes=NUM_CLASSES,
    device=device,
    model_load_path=model_save_path,
    visualize_samples=BATCH_SIZE, # Visualize 2 samples from the test set
    RGB_ToTrain=True,
    PD_ToTrain = DepthConstSet().raw
    # use_logits = True
)
